In [ ]:

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
DATASET_PATH = '../data/raw/student-mental-health.csv'
from sklearn.model_selection import train_test_split

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.append(str(PROJECT_ROOT))

In [ ]:
df_clean = pd.read_csv('../data/processed/cleaned_data.csv')
df_clean

In [ ]:
# separate features x and target y

X = df_clean.drop(columns= ['Mental_Health_Risk' , 'Student_ID'] )
y = df_clean['Mental_Health_Risk']

y

In [ ]:
# data splitting

X_train , X_test , y_train , y_test = train_test_split(X,
                                                       y,
                                                       test_size=0.2,
                                                       random_state=42,
                                                       stratify= y
                                                       )



In [ ]:
# find the min screen time value , to impute the missing values in Social_Media_Hours with it 
# j ai fais cette imputation apres splitting pour eviter le data leakage car  jai utiliser une imputaion qui utilise le min

min_screen_time = X_train['Screen_Time_Hours'].min()

X_train['Social_Media_Hours'] = X_train['Social_Media_Hours'].fillna(min_screen_time)

X_test['Social_Media_Hours'] = X_test['Social_Media_Hours'].fillna(min_screen_time)


In [ ]:
numerical_cols = X_train.select_dtypes(
    include=["int64", "float64"]
).columns

# iqr method

q1 = X_train[numerical_cols].quantile(0.25)
q3 = X_train[numerical_cols].quantile(0.75)

iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

iqr_outliers =( (X_train[numerical_cols] < lower_bound) | (X_train[numerical_cols] > upper_bound) )
print(iqr_outliers.sum().sum())


# z-score
mean = X_train[numerical_cols].mean()
std = X_train[numerical_cols].std()

z_scores = (X_train[numerical_cols] - mean) / std

zscore_outliers = z_scores.abs() > 3
print('---------')
print(zscore_outliers.sum().sum())

In [ ]:
import os
print(os.getcwd())

In [ ]:
# separation des types de variables
categorical_cols = X_train.select_dtypes(
    include= ['object', 'category']
).columns

numerical_cols = X_train.select_dtypes(include= np.number).columns
